# 第 02 章 质量控制与双细胞检测

## 学习目标

识别质量异常的细胞并了解双细胞检测，理解过滤阈值会影响后续分析。

## 为什么做与怎样做

从同一基础过滤对象分别生成 basic、mt、mad，比较后确认；再按捕获文库预测双细胞并确认保留或删除。

前置章节：01。运行前请完成项目环境准备。

本章在项目副本运行。遇到等待材料/确认，请按根目录 **99_运行与AI协作指南.md** 查看当前结果、与用户讨论并续跑；不能跳过待办直接进入下游。


In [ ]:
# 功能说明：查找公共环境和当前项目，读取有效上游检查点；模板副本之间不共享分析状态。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
from course_projects import resolve_project, input_path, read_sample
from course_resources import qc_gene_sets, marker_resources, resolution_preview
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
INSTALL_ROOT = ROOT
ROOT = resolve_project()
ctx = start_chapter("02")
adata = ctx.load_input()



## 02.1 预处理

In [ ]:
# 功能说明：设置 Scanpy 的图形输出目录。
# 运行目的：将生成的图片自动保存到指定文件夹，便于分类管理。
# 详细代码解析：
# 1. `sc.settings.figdir = ctx.figures`
#    - `sc`: Scanpy 库的别名。
#    - `.settings`: Scanpy 的全局设置对象。
#    - `.figdir`: 属性，指定保存图形（figure）的默认目录。
#    - `"本章结果目录"`: 相对路径字符串，指向当前目录下的 `figures` 文件夹内的 `1.预处理` 子文件夹。

# 设置输出目录
sc.settings.figdir = ctx.figures

每个观察值理想上对应一个完整单细胞。低质量细胞、双细胞以及环境 RNA 都可能影响这一假设。

本课程计算质量指标、执行基础过滤并检测双细胞；输入为过滤后的计数矩阵，主线不包含环境 RNA 去污染步骤。选择参数时需要兼顾数据质量与真实细胞异质性，避免过度过滤稀有群体。

计数矩阵中的零可能来自低表达、有限采样等原因，不能把所有零都判为技术缺失。条形码也不能直接等同于经过验证的单细胞。

## 02.2 质控

## 02.3 过滤低质量细胞

质量控制的第一步是从数据集中去除低质量细胞。
当一个细胞检测到的基因数量少、计数深度低且线粒体计数比例高时，它可能具有破损的膜，这可能表明细胞正在死亡。
由于这些细胞通常不是我们分析的主要目标，并且可能会扭曲我们的下游分析，因此我们在质量控制期间将其去除。
为了识别它们，我们定义了细胞质量控制 (QC) 阈值。
细胞 QC 通常在以下三个 QC 协变量上执行：

**条形码：是指测序时为每个细胞所做的标记编码，即理想情况下每个条形码对应一个细胞**
1. 每个条形码的计数数（计数深度）
2. 每个条形码的基因数
3. 每个条形码来自线粒体基因的计数比例

在细胞 QC 中，这些协变量通过阈值过滤，因为它们可能对应于死亡细胞。
如前所述，它们可能反映了膜破损的细胞，其细胞质 mRNA 已经泄漏，因此只有线粒体中的 mRNA 仍然存在。
这些细胞可能显示出低计数深度、检测到的基因少和线粒体读数比例高。
然而，至关重要的是要联合考虑这三个 QC 协变量，否则可能会导致对细胞信号的误解。
例如，具有相对较高比例线粒体计数的细胞可能参与呼吸过程，不应被过滤掉。
而计数低或高的细胞可能对应于静止细胞群或尺寸较大的细胞。
因此，在对单个协变量做出阈值决策时，首选考虑多个协变量。
通常，建议排除较少的细胞并**尽可能宽松**，以避免过滤掉可行的细胞群或小的亚群。

仅对少数或小型数据集进行的 QC 通常是通过手动查看不同 QC 协变量的分布并识别异常值来执行的，这些异常值随后将被过滤。
然而，随着数据集规模的增长，这项任务变得越来越耗时，可能值得考虑通过 MAD（中位数绝对偏差）进行自动阈值处理。
MAD 由 $MAD = median(|X_i - median(X)|)$ 给出，其中 $X_i$ 是观察值的相应 QC 指标，描述了指标变异性的稳健统计量。
与 （参考文献：qc:germain_pipecomp_2020） 类似，如果细胞相差 5 个 MAD，我们将细胞标记为异常值，这是一种相对宽松的过滤策略。
我们要强调的是，在细胞注释后重新评估过滤可能是合理的。


假设你测了 10 个细胞的 线粒体基因比例（%），排序后如下（单位：%）：
[2.1, 2.3, 2.5, 2.6, 2.7, 2.8, 3.0, 3.2, 4.5, 18.0]
计算中位数：第5、6个数的平均 = (2.7+2.8)/2 = 2.75%
计算每个值与中位数的绝对偏差：
|2.1-2.75|=0.65, 0.45, 0.25, 0.15, 0.05, 0.05, 0.25, 0.45, 1.75, 15.25
MAD = 这些绝对偏差的中位数 = 排序后 [0.05,0.05,0.15,0.25,0.25,0.45,0.45,0.65,1.75,15.25] → 中位数 = (0.25+0.45)/2 = 0.35%
设置不同倍数：
3 倍 MAD：阈值下限 = 2.75 - 3×0.35 = 1.70%，上限 = 2.75 + 3×0.35 = 3.80%
异常值：低于 1.70%？没有；高于 3.80% 的有 4.5% 和 18% → 标记 2 个细胞（严格）
5 倍 MAD：下限 = 2.75 - 5×0.35 = 1.00%，上限 = 2.75 + 1.75 = 4.50%
异常值：>4.50% 只有 18% → 标记 1 个细胞（宽松）
8 倍 MAD：上限 = 2.75 + 2.80 = 5.55%，只有 18% 仍为异常（更宽松）
结论：n=5 比 n=3 宽松，n=10 比 n=5 更宽松。

在 QC 中，第一步是计算 QC 协变量或指标。
我们使用 scanpy 函数 `sc.pp.calculate_qc_metrics` 计算这些指标，该函数还可以计算特定基因群体的计数比例。可以向 ~scanpy.pp.calculate_qc_metrics 传入特定基因集合，以计算这些集合在总计数中的占比。线粒体、核糖体与血红蛋白基因通常由特定前缀标识，如下所示。
因此，我们定义线粒体、核糖体和血红蛋白基因。
重要的是要注意，线粒体计数根据数据集中考虑的物种用前缀 "mt-" 或 "MT-" 注释。
如前所述，这里使用的数据集是人类骨髓，因此线粒体计数用前缀 "MT-" 注释。
对于小鼠数据集，前缀通常是小写的，即 "mt-"。

In [ ]:
# 功能说明：在基因注释表中标记线粒体、核糖体、血红蛋白相关基因。
# 运行目的：为 QC 指标计算提供基因集合（如线粒体占比）。
# 变量/函数/参数解析：
# - adata.var_names.str.startswith("MT-")：判断基因名是否以 "MT-" 开头（人类线粒体基因前缀）。
# - adata.var_names.str.startswith(("RPS", "RPL"))：判断是否以核糖体蛋白家族前缀 RPS/RPL 开头。
# - adata.var_names.str.contains("^HB[^(P)]")：使用正则匹配血红蛋白基因（排除 HBP 等）。
# - 结果列：在 `adata.var` 添加布尔列 `mt`、`ribo`、`hb` 表示该基因是否属于对应集合。
# 人类骨髓原示例（实际规则由项目登记）：adata.var["mt"] = adata.var_names.str.startswith("MT-")
# 核糖体基因
# 人类骨髓原示例（实际规则由项目登记）：adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# 血红蛋白基因
# 人类骨髓原示例（实际规则由项目登记）：adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
qc_vars = qc_gene_sets(ctx, adata)
qc_metrics = ["total_counts", "n_genes_by_counts"] + ["pct_counts_"+k for k in qc_vars]


In [ ]:
# 功能说明：查看 AnnData 对象的摘要信息。
# 运行目的：检查数据加载是否成功，查看细胞数（n_obs）、基因数（n_vars）及已有的注释信息。
# 详细代码解析：
# 1. `adata`
#    - 在 Jupyter Notebook 中直接输入变量名，会调用其 `__repr__` 方法，打印对象的概览。
#    - 输出通常包含：
#      - `n_obs × n_vars`: 细胞数 × 基因数。
#      - `obs`: 细胞的观测注释（如样本来源）。
#      - `var`: 基因的特征注释（如基因名）。

adata

In [ ]:
# 功能说明：查看基因注释表（var）。
# 运行目的：检查基因层面的元数据，确认是否成功添加了线粒体（mt）、核糖体（ribo）等标记列。
# 详细代码解析：
# 1. `adata.var`
#    - `adata`: AnnData 对象。
#    - `.var`: 存储变量（variable/基因）注释的 DataFrame（表格）。
#    - 行索引（index）通常是基因 ID 或基因名。
#    - 列（columns）包含基因的属性，如是否为线粒体基因 (`mt`)。

adata.var

In [ ]:
# 功能说明：查看细胞注释表（obs）。
# 运行目的：检查细胞层面的元数据，如样本来源（samples）等。
# 详细代码解析：
# 1. `adata.obs`
#    - `adata`: AnnData 对象。
#    - `.obs`: 存储观测（observation/细胞）注释的 DataFrame。
#    - 行索引（index）是细胞条形码（barcode）。
#    - 列（columns）包含细胞的属性，如所属样本。

adata.obs

我们现在可以使用 scanpy 计算相应的 QC 指标。

In [ ]:
# 功能说明：计算常见 QC 指标并写入 AnnData。
# 运行目的：获得每个细胞的线粒体/核糖体/血红蛋白占比、总计数、基因数等统计用于后续过滤与评估。
# 变量/函数/参数解析：
# - sc.pp.calculate_qc_metrics(adata, qc_vars=qc_vars, inplace=True, log1p=True)：
#   - adata(AnnData)：输入对象。
#   - qc_vars(list[str])：在 `adata.var` 中定义的布尔列名，表示基因集合。
#   - inplace(bool)：True 表示将结果直接写入 `adata.obs` 与 `adata.var`。
#   - log1p(bool)：对总计数等进行 `log1p`（log(1+x)）变换，得到额外列如 `log1p_total_counts`。
sc.pp.calculate_qc_metrics(adata, qc_vars=qc_vars, inplace=True, log1p=True)

sc.pp.calculate_qc_metrics函数向 `.var` 和 `.obs` 添加了几个额外的列。
我们想在这里重点介绍其中几个：

* `.obs` 中的 `n_genes_by_counts` 是细胞中具有正计数的基因数，
* `total_counts` 是细胞的总计数数，这也可能被称为文库大小
* `pct_counts_mt` 是细胞总计数中线粒体计数的比例。

我们现在绘制每个样本的三个 QC 协变量 `n_genes_by_counts`、`total_counts` 和 `pct_counts_mt`，以评估相应细胞的捕获情况。

现在可以用小提琴图检查部分 QC 指标：

- 表达基因数（每个细胞在计数矩阵中非零的基因数量）
- 每个细胞的总计数（UMI 总数）
- 线粒体基因占比（`pct_counts_mt`）

In [ ]:
# 功能说明：查看细胞注释表（obs）。
# 运行目的：检查细胞层面的元数据，如样本来源（samples）等。
# 详细代码解析：
# 1. `adata.obs`
#    - `adata`: AnnData 对象。
#    - `.obs`: 存储观测（observation/细胞）注释的 DataFrame。
#    - 行索引（index）是细胞条形码（barcode）。
#    - 列（columns）包含细胞的属性，如所属样本。

adata.obs

In [ ]:
# 功能说明：绘制多个 QC 指标的小提琴图以观察分布。
# 运行目的：直观比较细胞层面的质量控制指标，辅助设定过滤阈值。
# 变量/函数/参数解析：
# - sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)：
#   - adata：AnnData 对象。
#   - keys(list[str])：要绘制的 obs 列名。
#   - jitter(float)：散点抖动强度，便于显示密集数据点。
#   - multi_panel(bool)：True 表示多面板显示各指标。

sc.pl.violin(
    adata,
    qc_metrics,
    jitter=0.4,
    multi_panel=True,
    groupby='samples',
    save="_02_31.pdf",
)

In [ ]:
# 功能说明：绘制多个 QC 指标的小提琴图以观察分布。
# 运行目的：直观比较细胞层面的质量控制指标，辅助设定过滤阈值。
# 变量/函数/参数解析：
# - sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)：
#   - adata：AnnData 对象。
#   - keys(list[str])：要绘制的 obs 列名。
#   - jitter(float)：散点抖动强度，便于显示密集数据点。
#   - multi_panel(bool)：True 表示多面板显示各指标。

sc.pl.violin(
    adata,
    qc_metrics,
    jitter=0.4,
    multi_panel=True,
    save="_02_32.pdf",
)

此外，联合查看 QC 指标也很有用，例如绘制按 `pct_counts_mt` 着色的散点图，便于同时理解总计数与基因数与线粒体占比之间的关系。

In [ ]:
# 功能说明：散点图联合展示总计数与基因数，并以线粒体占比着色。
# 运行目的：评估高线粒体占比或异常总计数的细胞分布情况。
# 变量/函数/参数解析：
# - sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color=("pct_counts_mt" if "mt" in qc_vars else "total_counts"))：
#   - x(str)：`obs` 中作为横轴的列，此处为总计数。
#   - y(str)：`obs` 中作为纵轴的列，此处为基因数。
#   - color(str/list)：着色列，此处为线粒体占比。
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color=("pct_counts_mt" if "mt" in qc_vars else "total_counts"),save="_02_34.pdf")

联合查看各样本的总计数、检测基因数和线粒体占比，再评估过滤策略。默认仅移除检测基因数少于 100 的细胞，以及在少于 3 个细胞中出现的基因。

下面同时保留线粒体阈值和 MAD 两种可选策略。是否应用由课程参数控制；本次实际过滤数量会保存到表格。阈值不是适用于所有组织的固定标准。

## 02.4 手动设置阈值

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
qc_before = adata.obs.copy()
# 功能说明：按细胞与基因的简单阈值进行初步过滤。
# 运行目的：移除低质量细胞与低表达基因，减少噪音。
# 变量/函数/参数解析：
# - sc.pp.filter_cells(adata, min_genes=ctx.config["parameters"]["min_genes"])：
#   - min_genes(int)：每个细胞至少表达的基因数阈值；<100 的细胞将被移除。
# - sc.pp.filter_genes(adata, min_cells=ctx.config["parameters"]["min_cells"])：
#   - min_cells(int)：基因至少被多少细胞检测到；<3 的基因将被移除。

print(f"总细胞数: {adata.n_obs}")
sc.pp.filter_cells(adata, min_genes=ctx.config["parameters"]["min_genes"])
sc.pp.filter_genes(adata, min_cells=ctx.config["parameters"]["min_cells"])
print(f"过滤之后细胞数: {adata.n_obs}")



In [ ]:
# 变量/函数/参数解析：
# - adata_qc_base：基础过滤后的独立副本；MT 与 MAD 不在彼此结果上继续过滤。
# - ctx.candidate(stage, name, adata)：按阶段/候选保存对象与校验值，暂不形成主线检查点。
# 功能说明：保存共同基础对象，三个候选从相同输入出发。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
adata_qc_base = adata.copy()
qc_candidates = {"basic": ctx.candidate("qc", "basic", adata_qc_base, label="基础过滤")}


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if "mt" in qc_vars:
    adata_qc_mt = adata_qc_base.copy()
    # 功能说明：执行细胞过滤，移除线粒体基因标记为异常的细胞。
    # 运行目的：生成一个只包含高质量细胞的数据集，用于后续分析。
    # 详细代码解析：
    # 1. `print(f"总细胞数: {adata_qc_mt.n_obs}")`
    #    - 打印过滤前的总细胞数。
    # 2.  `adata_qc_mt[(adata_qc_mt.obs.pct_counts_mt <= 20)]`:
    #    使用布尔索引对AnnData对象进行切片操作：
    #    - 只保留索引为True的细胞(行)
    #    - 丢弃索引为False的细胞(行)
    # 3. `print(f"过滤之后细胞数: {adata_qc_mt.n_obs}")`
    #    - 打印过滤后的细胞数，以评估过滤掉的比例。

    print(f"总细胞数: {adata_qc_mt.n_obs}")
    adata_qc_mt = adata_qc_mt[(adata_qc_mt.obs.pct_counts_mt<=ctx.config["qc"]["mt_cutoff"])].copy()

    print(f"过滤之后细胞数: {adata_qc_mt.n_obs}")

    qc_candidates["mt"] = ctx.candidate("qc", "mt", adata_qc_mt, label="基础加线粒体阈值", cutoff=ctx.config["qc"]["mt_cutoff"])



## 02.5 自动设置阈值

以下保留基于 MAD 的自动阈值策略。函数按样本分别计算中位数和 MAD；默认主线不应用该过滤。启用后应比较每个样本的保留情况。

In [ ]:
# 按样本计算中位数和 MAD，返回每个细胞的异常标记。
def is_outlier(adata, metric: str, nmads: int):
    values = adata.obs[metric]
    group = adata.obs["samples"]
    center = values.groupby(group, observed=True).transform("median")
    deviation = values.groupby(group, observed=True).transform(
        lambda x: median_abs_deviation(x, nan_policy="omit")
    )
    # MAD 为 0 时不因浮点误差自动标记异常，交由联合 QC 检查。
    return ((values - center).abs() > nmads * deviation) & (deviation > 0)

In [ ]:
# 功能说明：MAD 使用相同基础对象，不以线粒体候选的输出作为输入。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
adata_qc_mad = adata_qc_base.copy()


启用 MAD 策略时，联合检查 log1p_total_counts、log1p_n_genes_by_counts 和 pct_counts_in_top_50_genes，各使用 5 个 MAD。

In [ ]:
# 功能说明：应用异常值检测函数，识别低质量细胞。
# 运行目的：基于多个指标（总计数、基因数、高表达基因占比）综合标记异常细胞。
# 详细代码解析：
# 1. `adata_qc_mad.obs["outlier"] = (...)`
#    - 在 adata_qc_mad.obs 中创建一个名为 "outlier" 的新列。
# 2. `is_outlier(adata_qc_mad, "log1p_total_counts", ctx.config["qc"]["general_mads"])`
#    - 检测 log1p 变换后的总计数是否偏离 5 个 MAD。
# 3. `| is_outlier(adata_qc_mad, "log1p_n_genes_by_counts", ctx.config["qc"]["general_mads"])`
#    - 或者，检测 log1p 变换后的基因数是否偏离 5 个 MAD。
# 4. `| is_outlier(adata_qc_mad, "pct_counts_in_top_20_genes", ctx.config["qc"]["general_mads"])`
#    - 或者，检测前 20 个基因的表达占比是否偏离 5 个 MAD。
# 5. `adata_qc_mad.obs.outlier.value_counts()`
#    - 统计 "outlier" 列中 True（异常）和 False（正常）的数量，查看有多少细胞被标记为异常。

adata_qc_mad.obs["outlier"] = (
    is_outlier(adata_qc_mad, "log1p_total_counts", ctx.config["qc"]["general_mads"])
    | is_outlier(adata_qc_mad, "log1p_n_genes_by_counts", ctx.config["qc"]["general_mads"])
    | is_outlier(adata_qc_mad, "pct_counts_in_top_50_genes", ctx.config["qc"]["general_mads"])
)
adata_qc_mad.obs.outlier.value_counts()


启用 MAD 策略时，另以 3 个 MAD 与线粒体比例 20% 的阈值标记线粒体指标异常。阈值需结合组织和分布评估。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
if "mt" in qc_vars:
    # 功能说明：专门检测线粒体比例异常的细胞。
    # 运行目的：线粒体比例过高通常意味着细胞破损，需要单独严格过滤。
    # 详细代码解析：
    # 1. `adata_qc_mad.obs["mt_outlier"] = ...`
    #    - 创建 "mt_outlier" 列标记线粒体异常细胞。
    # 2. `is_outlier(adata_qc_mad, "pct_counts_mt", ctx.config["qc"]["mt_mads"])`
    #    - 基于 MAD 检测异常值（阈值为 3 MAD）。
    # 3. `| (adata_qc_mad.obs["pct_counts_mt"] > ctx.config["qc"]["mt_cutoff"])`
    #    - 或者，硬性规定线粒体比例大于 20% 的也算异常。这是结合了自动检测和先验知识的策略。
    # 4. `adata_qc_mad.obs.mt_outlier.value_counts()`
    #    - 统计线粒体异常细胞的数量。

    adata_qc_mad.obs["mt_outlier"] = is_outlier(adata_qc_mad, "pct_counts_mt", ctx.config["qc"]["mt_mads"]) | (
        adata_qc_mad.obs["pct_counts_mt"] > ctx.config["qc"]["mt_cutoff"]
    )
    adata_qc_mad.obs.mt_outlier.value_counts()

else:
    adata_qc_mad.obs["mt_outlier"] = False



启用 MAD 策略时，根据两个异常标记过滤细胞。默认主线保留这些细胞，供后续聚类和标记基因复核。

In [ ]:
# 功能说明：执行细胞过滤，移除标记为异常的细胞。
# 运行目的：生成一个只包含高质量细胞的数据集，用于后续分析。
# 详细代码解析：
# 1. `print(f"总细胞数: {adata_qc_mad.n_obs}")`
#    - 打印过滤前的总细胞数。
# 2. `adata_qc_mad = adata_qc_mad[(~adata_qc_mad.obs.outlier) & (~adata_qc_mad.obs.mt_outlier)].copy()`
#    - `~adata_qc_mad.obs.outlier`: 取反，选择非通用异常值的细胞。
#    - `~adata_qc_mad.obs.mt_outlier`: 取反，选择非线粒体异常值的细胞。
#    - `&`: 逻辑与，必须同时满足两个条件。
#    - `adata_qc_mad[...]`: 切片操作，保留满足条件的细胞。
#    - `.copy()`: 创建一个新的 AnnData 对象副本，断开与原对象的内存连接，释放空间。
# 3. `print(f"过滤之后细胞数: {adata_qc_mad.n_obs}")`
#    - 打印过滤后的细胞数，以评估过滤掉的比例。


print(f"总细胞数: {adata_qc_mad.n_obs}")
adata_qc_mad = adata_qc_mad[(~adata_qc_mad.obs.outlier) & (~adata_qc_mad.obs.mt_outlier)].copy()
print(f"过滤之后细胞数: {adata_qc_mad.n_obs}")


In [ ]:
# 变量/函数/参数解析：
# - ctx.choose_data：先导出各候选的数量、保留比例、分布和交集，再等待真实确认。
# - 返回值：已确认候选的独立 AnnData；下游只消费这个对象。
# - 再次运行：同一输入复核并使用原候选；改变科学输入后必须新开尝试。
# 功能说明：先保存和比较过滤候选，用户确认后才检测双细胞。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
qc_candidates["mad"] = ctx.candidate("qc", "mad", adata_qc_mad, label="基础加 MAD（有 mt 时含其上限）", rules=ctx.config["qc"])
adata = ctx.choose_data("qc", qc_candidates, "请先比较按样本统计和 QC 图，解释不同过滤规则的代价，再确认 basic、mt 或 mad。")


In [ ]:
# 功能说明：过滤后再次绘制散点图。
# 运行目的：验证过滤效果，确保异常点（如高线粒体比例、极低/极高计数的细胞）已被移除。
# 详细代码解析：
# 1. `p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color=("pct_counts_mt" if "mt" in qc_vars else "total_counts"))`
#    - 再次绘制总计数 vs 基因数的散点图，并用线粒体比例着色。
#    - 预期结果：图中的点分布应更加集中，颜色（线粒体比例）应普遍较低。
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color=("pct_counts_mt" if "mt" in qc_vars else "total_counts"),save="_02_48.pdf")


## 02.6 双细胞（Doublet）检测

下一步运行双细胞检测算法。识别双细胞很重要，因为它们可能导致后续分析的误分类或偏差。Scanpy 提供 Scrublet 方法，其通过邻近分类器在观测转录组与模拟双细胞之间进行预测。scanpy.pp.scrublet 会在 `.obs` 添加 `doublet_score` 和 `predicted_doublet`。


双细胞定义为在同一细胞条形码下测序的两个细胞，例如，如果它们被捕获在同一个液滴中。
这就是为什么我们直到现在都使用术语“条形码”而不是“细胞”。
如果双细胞由相同的细胞类型形成，则称为同型双细胞，否则称为异型双细胞。
同型双细胞不一定能从计数矩阵中识别出来，仍可能影响计数和比例；细胞哈希或 SNP 在相应实验条件下可提供补充证据。
因此，它们的识别不是双细胞检测方法的主要目标。

由不同细胞类型或状态形成的双细胞称为异型双细胞。
它们的识别至关重要，因为它们最有可能被错误分类，并可能导致下游分析步骤失真。
因此，双细胞检测和去除通常是初始预处理步骤。
双细胞可以通过其大量的读取和检测到的特征来识别，也可以通过创建人工双细胞并将其与数据集中存在的细胞进行比较的方法来识别。

在 scverse 生态中，其他双细胞检测方法包括 [DoubletDetection](https://github.com/JonathanShor/DoubletDetection) 与 [SOLO](https://docs.scvi-tools.org/en/stable/user_guide/models/solo.html)。更多内容可参考 Single Cell Best Practices 的[双细胞检测章节](https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html#doublet-detection)。


我们可以通过两种方式去除双细胞：要么直接过滤掉被判定为双细胞的细胞，要么在完成一次聚类后，过滤掉双细胞分数高的簇。



**之后可以直接根据 `predicted_doublet` 过滤，或在聚类后根据 `doublet_score` 去除双细胞比例较高的簇。**



我建议暂时将识别出的双细胞保留在数据集中，并在后续可视化期间检查双细胞。

In [ ]:
# 变量/函数/参数解析：
# - doublet_cache：当前 QC 选择对应的预测缓存，帮助恢复同一次教学会话。
# - capture_library：一次细胞捕获文库；不同测序 lane 是否同库需根据实验记录判断。
# - predicted_doublet 只是模型预测；是否删除需比较两种候选并由用户确认。
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
doublet_cache = ctx.directory / ("doublet_" + ctx.record["decisions"]["qc"]["candidate"] + ".h5ad")
if doublet_cache.exists():
    adata = sc.read_h5ad(doublet_cache)
else:
    # 功能说明：运行 Scrublet 双细胞检测并按批次处理。
    # 运行目的：为每个细胞计算双细胞分数并预测是否为双细胞。
    # 变量/函数/参数解析：
    # - sc.pp.scrublet(adata, batch_key="capture_library")：
    #   - batch_key(str)：按 `obs['sample']` 分批运行，避免批次差异影响检测。
    # 输出：在 `adata.obs` 增加 `doublet_score`（浮点分数）与 `predicted_doublet`（布尔预测）。
    sc.pp.scrublet(adata, batch_key="capture_library")
    adata.write_h5ad(doublet_cache, compression="gzip")



In [ ]:
# 功能说明：查看双细胞预测结果列。
# 运行目的：检查 Scrublet 算法生成的双细胞预测标签（True/False）。
# 详细代码解析：
# 1. `adata.obs["predicted_doublet"]`
#    - `adata.obs`: 细胞注释 DataFrame。
#    - `["predicted_doublet"]`: 选择名为 "predicted_doublet" 的列。
#    - 该列由 `sc.pp.scrublet` 生成，包含布尔值（True 表示预测为双细胞，False 表示单细胞）。

adata.obs["predicted_doublet"]

In [ ]:
# 变量/函数/参数解析：
# - ctx.choose_data：先导出各候选的数量、保留比例、分布和交集，再等待真实确认。
# - 返回值：已确认候选的独立 AnnData；下游只消费这个对象。
# - 再次运行：同一输入复核并使用原候选；改变科学输入后必须新开尝试。
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
adata_without_doublets = adata.copy()

# 功能说明：执行细胞过滤，在运行了 scrublet 之后，进行过滤
# 运行目的：生成一个只包含高质量细胞的数据集，用于后续分析。
# 详细代码解析：
# 1. `print(f"Total number of cells: {adata_without_doublets.n_obs}")`
#    - 打印过滤前的总细胞数。

# 2.  adata_without_doublets.obs[‘predicted_doublet’]：是布尔值序列。
# ~ 符号：代表逻辑“非”，即取反。~True 是 False，~False 是 True。
# 所以，~adata_without_doublets.obs[‘predicted_doublet’] 就选中了所有 不是 (False) 双细胞的细胞。
# adata_without_doublets = adata_without_doublets[筛选条件, :]：通过这个索引操作，只保留符合条件的细胞（即非双细胞）。
# .copy()：为了避免后续操作可能出现的警告，建议创建一个新的数据副本。

# 3. `print(f"Number of cells after filtering of low quality cells: {adata_without_doublets.n_obs}")`
#    - 打印过滤后的细胞数，以评估过滤掉的比例。

print(f"总细胞数: {adata_without_doublets.n_obs}")
adata_without_doublets = adata_without_doublets[~adata_without_doublets.obs.predicted_doublet].copy()
print(f"过滤之后细胞数: {adata_without_doublets.n_obs}")

doublet_candidates = {"keep": ctx.candidate("doublets", "keep", adata, label="保留并标记预测双细胞"), "remove": ctx.candidate("doublets", "remove", adata_without_doublets, label="删除预测双细胞")}
adata = ctx.choose_data("doublets", doublet_candidates, "请比较分文库预测数量、分数和去除后的保留情况，确认 keep 或 remove。")



## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
qc_metrics = ["total_counts", "n_genes_by_counts"] + ["pct_counts_"+k for k in qc_vars]
ctx.table("cell_qc", adata.obs)
qc_summary = adata.obs.groupby("samples", observed=True)[qc_metrics].agg(["median", "mean", "min", "max"])
ctx.table("qc_by_sample", qc_summary)
retention = pd.DataFrame({"before": qc_before["samples"].value_counts(), "after": adata.obs["samples"].value_counts()}).fillna(0).astype(int)
retention["removed"] = retention["before"] - retention["after"]
retention["retained_fraction"] = retention["after"] / retention["before"]
ctx.table("cell_retention", retention)
doublets = adata.obs.groupby("samples", observed=True)["predicted_doublet"].agg(["sum", "count", "mean"])
ctx.table("doublets_by_sample", doublets)
ctx.finish(adata, {"retention": retention.to_dict(orient="index"), "predicted_doublets": int(adata.obs["predicted_doublet"].sum()), "qc_metrics_basis": "过滤前的基因集合", "doublets_removed": ctx.record["decisions"]["doublets"]["candidate"] == "remove"})


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：线粒体比例高，是否足以单独证明一个细胞必须被移除？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。